# Chapter 2: LLM API Basics

Companion notebook for Chapter 2. Each section corresponds to a listing in the book.

Run the code cells from top to bottom in a fresh kernel after completing the
README setup and this chapter's API prerequisites.


## Setup (Listing 2.1)

In [1]:
from pathlib import Path
from dotenv import load_dotenv

# Locate this checkout, including when Jupyter starts in a chapter folder.
PROJECT_ROOT = next(
    p for p in (Path.cwd(), *Path.cwd().parents)
    if (p / "pyproject.toml").is_file() and (p / "scratch_agents").is_dir()
)
import sys
# Use this checkout even if the selected kernel was installed elsewhere.
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
load_dotenv(PROJECT_ROOT / ".env")

%load_ext autoreload
%autoreload 2


## OpenAI API (Listing 2.2)

In [8]:
import os
import uuid
from openai import OpenAI

client = OpenAI(
    api_key=os.environ["DEEPSEEK_API_KEY"],
    base_url="https://api.deepseek.com",
)

response = client.chat.completions.create(
    model="deepseek-flash",
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "What is the capital of France?"},
    ],
)

print(response.choices[0].message.content)

The capital of France is **Paris**.


## Anthropic API (Listing 2.3)

In [ ]:
from anthropic import Anthropic

client = Anthropic()

response = client.messages.create(
    model="claude-sonnet-4-6",
    max_tokens=1024,
    messages=[
        {"role": "user", "content": "What is the capital of France?"}
    ]
)

print(response.content[0].text)

## Unified API with LiteLLM (Listing 2.4)

In [10]:
import os
from litellm import completion

response = completion(
    model="deepseek/deepseek-flash",
    api_key=os.environ["DEEPSEEK_API_KEY"],
    messages=[{"role": "user", "content": "Hello!"}],
    extra_body={"thinking": {"type": "disabled"}},
)
print(response.choices[0].message.content)

response = completion(
    model="deepseek/deepseek-v4-pro",
    api_key=os.environ["DEEPSEEK_API_KEY"],
    messages=[{"role": "user", "content": "Hello!"}],
    extra_body={"thinking": {"type": "disabled"}},
)
print(response.choices[0].message.content)

Hello! How can I help you today?
Hi there! How can I help you today?


## Demonstrating API statelessness (Listing 2.5)

In [11]:
import os
from litellm import completion

def ask(messages):
    return completion(
        model="deepseek/deepseek-flash",
        api_key=os.environ["DEEPSEEK_API_KEY"],
        messages=messages,
        extra_body={"thinking": {"type": "disabled"}},
    )

# First call
response1 = ask([{"role": "user", "content": "My name is Jungjun."}])
print(response1.choices[0].message.content)

# Second call does not include the first message, so the API has no memory of it.
response2 = ask([{"role": "user", "content": "What is my name?"}])
print(response2.choices[0].message.content)

Hello, Jungjun! It's nice to meet you. How can I help you today?
I don't know your name — you haven't told me yet, and I don't have access to any personal information about you. If you'd like, you can tell me what you'd like to be called, and I'll use that in our conversation!


## Managing conversation history (Listing 2.6)

In [13]:
from litellm import completion

messages = []

# First exchange
messages.append({"role": "user", "content": "My name is Jungjun."})
response1 = completion(model="deepseek/deepseek-flash", messages=messages)
assistant_message1 = response1.choices[0].message.content
messages.append({"role": "assistant", "content": assistant_message1})
print(assistant_message1)

# Second exchange - includes previous conversation history
messages.append({"role": "user", "content": "What is my name?"})
response2 = completion(model="deepseek/deepseek-flash", messages=messages)
assistant_message2 = response2.choices[0].message.content
print(assistant_message2)

Nice to meet you, Jungjun! How can I help you today?
Your name is Jungjun.


## Structured output with Pydantic (Listing 2.7)

In [15]:
from pydantic import BaseModel
from litellm import completion

class ExtractedInfo(BaseModel):
    name: str
    email: str
    phone: str | None = None

response = completion(
    model="deepseek/deepseek-flash",
    messages=[
        {
            "role": "system",
            "content": (
                "Return JSON with keys name, email, and phone. "
                'Example: {"name": "Ada Lovelace", "email": "ada@example.com", "phone": "555-0100"}'
            ),
        },
        {
            "role": "user",
            "content": "My name is John Smith, my email is john@example.com, and my phone is 555-1234.",
        },
    ],
    response_format={"type": "json_object"},
    extra_body={"thinking": {"type": "disabled"}},
)

result = ExtractedInfo.model_validate_json(response.choices[0].message.content)
print(result)

name='John Smith' email='john@example.com' phone='555-1234'


## Asynchronous LLM calls (Listing 2.8)

In [16]:
import asyncio
from litellm import acompletion

async def get_response(prompt: str) -> str:
    response = await acompletion(
        model="deepseek/deepseek-flash",
        messages=[{"role": "user", "content": prompt}]
    )
    return response.choices[0].message.content

prompts = [
    "What is 2 + 2?",
    "What is the capital of Japan?",
    "Who wrote Romeo and Juliet?"
]

# Execute all requests concurrently
tasks = [get_response(p) for p in prompts]
results = await asyncio.gather(*tasks)

for prompt, result in zip(prompts, results):
    print(f"Q: {prompt}")
    print(f"A: {result}\n")

Q: What is 2 + 2?
A: 4

Q: What is the capital of Japan?
A: The capital of Japan is **Tokyo**.

Q: Who wrote Romeo and Juliet?
A: William Shakespeare wrote *Romeo and Juliet*.



## Rate-limited concurrent requests (Listing 2.9)

In [17]:
import asyncio
from litellm import acompletion

# Limit to 10 concurrent requests
semaphore = asyncio.Semaphore(10)

async def call_llm(prompt: str) -> str:
    """LLM call with rate limiting and automatic retry."""
    async with semaphore:
        response = await acompletion(
            model="deepseek/deepseek-flash",
            messages=[{"role": "user", "content": prompt}],
            num_retries=3  # Automatic retry with exponential backoff
        )
        return response.choices[0].message.content

# Even with 100 concurrent tasks, only 10 API calls run at a time
prompts = [f"What is {i} + {i}?" for i in range(100)]
tasks = [call_llm(p) for p in prompts]
results = await asyncio.gather(*tasks, return_exceptions=True)

successes = sum(1 for r in results if not isinstance(r, Exception))
print(f"Successful: {successes}/{len(results)}")

Successful: 100/100


## GAIA Benchmark Evaluation

Listings 2.10–2.17 are implemented in `scratch_agents.eval.gaia`. The notebook
imports the orchestration function and runs the experiment. Read the source
of `scratch_agents/eval/gaia.py` alongside the book listings for the full
implementation.

### Loading GAIA Level 1 problems (Listing 2.10)

In [2]:
# GAIA is gated: accept access on Hugging Face and set HF_TOKEN first.
from datasets import load_dataset

level1_problems = load_dataset("gaia-benchmark/GAIA", "2023_level1", split="validation")
print(f"Number of Level 1 problems: {len(level1_problems)}")

c:\Users\peter\ai-agent-from-scratch\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Number of Level 1 problems: 53


### Model selection and execution (Listing 2.17)

In [3]:
from scratch_agents.eval.gaia import run_experiment

MODELS = [
    "deepseek/deepseek-flash",
    "deepseek/deepseek-v4-pro"
]

subset = level1_problems.select(range(20))
results = await run_experiment(subset, MODELS)

# Per-model accuracy
for model, model_results in results.items():
    correct = sum(1 for r in model_results if r["correct"])
    solvable = sum(1 for r in model_results if r.get("is_solvable"))
    total = len(model_results)
    print(f"{model}: {correct}/{total} correct, {solvable}/{total} judged solvable")

100%|██████████| 40/40 [00:16<00:00,  2.36it/s]

deepseek/deepseek-flash: 2/20 correct, 9/20 judged solvable
deepseek/deepseek-v4-pro: 1/20 correct, 9/20 judged solvable
